# Quantum Fourier Transform (QFT)

The Quantum Fourier Transform is the quantum analogue of the Discrete Fourier
Transform (DFT).  It is a key subroutine in many quantum algorithms including
Shor's factoring algorithm and quantum phase estimation.

## Definition

The QFT on $n$ qubits maps a computational basis state $|j\rangle$ to:

$$\text{QFT}\,|j\rangle
  = \frac{1}{\sqrt{2^n}} \sum_{k=0}^{2^n-1} e^{2\pi i jk/2^n}\,|k\rangle$$

This is exactly the DFT matrix $F_{jk} = e^{2\pi i jk/N}/\sqrt{N}$ applied to
the amplitude vector.  The QFT is **unitary**: $FF^\dagger = I$.

## Circuit (4-qubit example)

```
q0 ----H--R2--R3--R4-----------------x---
q1 --------o--H--R2--R3-----------x--|
q2 ------------o--o--H--R2-----x--|--|
q3 -----------------o--o--H-x--|--|--|
                              x--x  x--x
```

Reading left to right for the most-significant qubit (qubit 3 in SymPy
right-to-left indexing):

```
l0 ---------------o-----------o-------o---H--x---
l1 -----------o---|--------o--|---H--R2------|--x
l2 --------o--|---|----H--R2--R3-------------|--x
l3 ----H--R2--R3--R4-------------------------x---
```

## Gates

- $H = \dfrac{1}{\sqrt{2}}\begin{bmatrix}1&1\\1&-1\end{bmatrix}$ (Hadamard)

- $R_k = \begin{bmatrix}1&0\\0&e^{2\pi i/2^k}\end{bmatrix}$ (controlled phase rotation by $2\pi/2^k$)

Each qubit receives a Hadamard followed by controlled-$R_k$ rotations from
more-significant qubits.  A final layer of SWAP gates reverses the qubit
order to match the standard QFT output convention.

## Complexity

The circuit uses $O(n^2)$ gates, compared to the classical FFT's $O(n 2^n)$
operations.  This exponential saving is exploited in phase estimation and
Shor's algorithm.

## Imports

In [1]:
from sympy import I, pi, exp, ImmutableMatrix
from sympy.physics.quantum.qapply import qapply
from sympy.physics.quantum.qubit import Qubit

from sympy.physics.quantum.gate import HadamardGate
from sympy.physics.quantum.gate import UGate, SwapGate, CGate
from sympy.physics.quantum.qft import QFT

## QFT circuit builder

The function constructs the QFT circuit as a product of SymPy gate objects.
The build order processes qubits from the most-significant to the
least-significant (reversed range), matching the standard QFT circuit layout:

1. For each qubit `line` (high to low): apply $H$, then a sequence of
   controlled-$R_k$ gates where each $R_k$ is controlled by a lower-indexed
   (more-significant) qubit.
2. Append SWAP gates to reverse the bit order of the output.

In [2]:
def qft(begin, end):
    """
    Build the QFT circuit acting on qubits begin..end-1.

    The circuit is returned as a SymPy gate product that can be applied to
    a Qubit state with qapply().  It matches SymPy's built-in QFT(begin, end)
    decomposition.

    Parameters
    ----------
    begin : int -- index of the first (lowest) qubit in the target register
    end   : int -- one past the index of the last qubit (exclusive upper bound)

    Returns
    -------
    SymPy gate product representing the full QFT circuit on qubits [begin, end).

    Notes
    -----
    SymPy indexes qubits right-to-left, so qubit 0 is the least-significant bit.
    The loop processes qubits from high index (most-significant) to low index
    (least-significant) to build the standard QFT column-by-column.
    """
    circuit = 1

    # process each qubit from most-significant to least-significant
    for line in reversed(range(begin, end)):
        # apply Hadamard to put this qubit into superposition
        circuit = HadamardGate(line) * circuit

        # apply controlled-R_k rotations from lower-indexed (higher-significance) qubits
        for i in range(line - begin):
            # rotation order: R_2 first, then R_3, R_4, ...
            n = i + 2
            # phase gate: diagonal [1, exp(2*pi*i / 2^n)]
            phase_gate = UGate(
                line,
                ImmutableMatrix([[1, 0], [0, exp(2 * pi * I / (2**n))]])
            )
            # control is on qubit (line - i - 1), which is more significant
            circuit = CGate(line - i - 1, phase_gate) * circuit

    # swap gates reverse the qubit order to match the standard QFT output
    for i in range((end - begin) // 2):
        circuit = SwapGate(i + begin, end - i - 1) * circuit

    return circuit

## Tests

Both tests verify the custom `qft` implementation against SymPy's built-in
`QFT(begin, end).decompose()`.  They apply the circuit to a fixed input state
and assert the outputs are equal.

### test_0 -- 4-qubit QFT on $|0101\rangle$

The input $|0101\rangle$ represents the integer 5.  The QFT maps it to a
superposition with amplitudes $e^{2\pi i \cdot 5k/16}/4$ for $k = 0,\ldots,15$.

In [3]:
def test_0():
    n = 4
    circ = qft(0, n)
    print(f"\nQFT circuit (n={n}): {circ}")

    q = Qubit("0101")  # integer 5 in 4-bit binary
    r = qapply(circ * q)

    # compare against SymPy's reference QFT decomposition
    qft_ref = qapply(QFT(0, n).decompose() * q)

    assert r == qft_ref


test_0()


QFT circuit (n=4): SWAP(1,2)*SWAP(0,3)*H(0)*C((0),U((1,),Matrix([
[1, 0],
[0, I]])))*H(1)*C((0),U((2,),Matrix([
[1,           0],
[0, exp(I*pi/4)]])))*C((1),U((2,),Matrix([
[1, 0],
[0, I]])))*H(2)*C((0),U((3,),Matrix([
[1,           0],
[0, exp(I*pi/8)]])))*C((1),U((3,),Matrix([
[1,           0],
[0, exp(I*pi/4)]])))*C((2),U((3,),Matrix([
[1, 0],
[0, I]])))*H(3)


### test_1 -- 6-qubit QFT on $|010101\rangle$

The input $|010101\rangle$ represents the integer 21.  The 6-qubit QFT
operates on a $2^6 = 64$-dimensional space and uses $O(6^2) = 36$ gates.

In [4]:
def test_1():
    n = 6
    circ = qft(0, n)
    print(f"\nQFT circuit (n={n}): {circ}")

    q = Qubit("010101")  # integer 21 in 6-bit binary
    r = qapply(circ * q)

    # compare against SymPy's reference QFT decomposition
    qft_ref = qapply(QFT(0, n).decompose() * q)

    assert r == qft_ref


test_1()


QFT circuit (n=6): SWAP(2,3)*SWAP(1,4)*SWAP(0,5)*H(0)*C((0),U((1,),Matrix([
[1, 0],
[0, I]])))*H(1)*C((0),U((2,),Matrix([
[1,           0],
[0, exp(I*pi/4)]])))*C((1),U((2,),Matrix([
[1, 0],
[0, I]])))*H(2)*C((0),U((3,),Matrix([
[1,           0],
[0, exp(I*pi/8)]])))*C((1),U((3,),Matrix([
[1,           0],
[0, exp(I*pi/4)]])))*C((2),U((3,),Matrix([
[1, 0],
[0, I]])))*H(3)*C((0),U((4,),Matrix([
[1,            0],
[0, exp(I*pi/16)]])))*C((1),U((4,),Matrix([
[1,           0],
[0, exp(I*pi/8)]])))*C((2),U((4,),Matrix([
[1,           0],
[0, exp(I*pi/4)]])))*C((3),U((4,),Matrix([
[1, 0],
[0, I]])))*H(4)*C((0),U((5,),Matrix([
[1,            0],
[0, exp(I*pi/32)]])))*C((1),U((5,),Matrix([
[1,            0],
[0, exp(I*pi/16)]])))*C((2),U((5,),Matrix([
[1,           0],
[0, exp(I*pi/8)]])))*C((3),U((5,),Matrix([
[1,           0],
[0, exp(I*pi/4)]])))*C((4),U((5,),Matrix([
[1, 0],
[0, I]])))*H(5)
